In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", message=".*Ignoring cached namespace.*") # Stops annoying library version warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

In [ ]:
output_dir = '/Users/lyn/Documents/UCSD Stuffs/DSC 180A/allensdk-data' # must be updated to a valid directory in your filesystem
DOWNLOAD_COMPLETE_DATASET = True

manifest_path = os.path.join(output_dir, "manifest.json")

cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)

In [ ]:
sessions = cache.get_session_table()

print('Total number of sessions: ' + str(len(sessions)))

sessions.head()

In [ ]:
filtered_sessions = sessions[(sessions.session_type == 'brain_observatory_1.1') & \
                             (['VISl' in acronyms for acronyms in sessions.ecephys_structure_acronyms])]

filtered_sessions.head()

In [ ]:
session_id1 = 715093703
session1 = cache.get_session_data(session_id1)
session1.metadata

In [ ]:
session_id2 = 750749662
session2 = cache.get_session_data(session_id2)
session2.metadata

In [ ]:
def get_units(session, stim="drifting_gratings", region="VISp"):
    stim_presentations = session.get_stimulus_table(stim)
    units = session.units[session.units["ecephys_structure_acronym"] == region]
    return stim_presentations, units

grating_presentations, v1_units = get_units(session1, region="VISp")
v1_units

In [ ]:
time_step = 0.01
time_bins = np.arange(-0.1, 0.5 + time_step, time_step)

histograms = session.presentationwise_spike_counts(
    stimulus_presentation_ids=grating_presentations.index.values,  
    bin_edges=time_bins,
    unit_ids=v1_units.index.values
)

histograms.coords

mean_histograms = histograms.mean(dim="stimulus_presentation_id")

fig, ax = plt.subplots(figsize=(8, 8))
ax.pcolormesh(
    mean_histograms["time_relative_to_stimulus_onset"], 
    np.arange(mean_histograms["unit_id"].size),
    mean_histograms.T, 
    vmin=0,
    vmax=1
)

ax.set_ylabel("unit", fontsize=18)
ax.set_xlabel("time relative to stimulus onset (s)", fontsize=18)
ax.set_title("peristimulus time histograms for VISp units on drifting gratings", fontsize=24)

plt.show()

In [ ]:
def get_spikes(session, stim_presentations, units):
    spikes = session.presentationwise_spike_times(
        stimulus_presentation_ids=stim_presentations.index.values,
        unit_ids=units.index.values[:]
    )
    spikes = spikes.reset_index()
    return spikes

spikes = get_spikes(session, grating_presentations, v1_units)
spikes

In [ ]:
spikes['time_since_stimulus_presentation_onset'].hist()

In [ ]:
def get_grating_angles(grating_presentations):
    grating_angles = grating_presentations["orientation"].reset_index()
    return grating_angles
grating_angles = get_grating_angles(grating_presentations)
grating_angles

In [ ]:
def clean_grating_spike_data(grating_spikes, grating_angles, window_size=0.2):
    grate_angle_spikes = spikes.merge(grating_angles, on="stimulus_presentation_id")
    fast_spikes_clean = grate_angle_spikes[(grate_angle_spikes["orientation"] != "null") & \
                                           (grate_angle_spikes["time_since_stimulus_presentation_onset"] <= window_size)]
    fast_spikes_clean = fast_spikes_clean.dropna()
    return fast_spikes_clean

s = 0.2
fast_spikes_clean = clean_grating_spike_data(spikes, grating_angles, window_size=s)
fast_spikes_clean

In [ ]:
# Step 1: compute rate per (trial, unit)
firing_rates = (
    fast_spikes_clean
    .groupby(['stimulus_presentation_id', 'unit_id'])
    .size()
    .div(s)
    .rename('rate_hz')
    .reset_index()
)
firing_rates = firing_rates.merge(grating_angles, on="stimulus_presentation_id")

# Step 2: average across trials for each orientation
orientation_tuning = (
    firing_rates.groupby('orientation')['rate_hz']
    .mean()
    .reset_index()
)

# Step 3: calculate 95% CIs
summary = (
    firing_rates
    .groupby('orientation')['rate_hz']
    .agg(['mean', 'std', 'count'])
    .sort_values(by='orientation')
    .reset_index()
)
summary['ci95'] = 1.96 * summary['std'] / np.sqrt(summary['count'])

# Step 4: plot avg firing rates against grating angle
plt.plot(orientation_tuning['orientation'], orientation_tuning['rate_hz'], 'o-')
plt.errorbar(
    summary['orientation'],
    summary['mean'],
    yerr=summary['ci95'],
    fmt='o-',
    color='navy',
    ecolor='crimson',
    capsize=4,
    label='mean ± 95% CI'
)

plt.title('V1 Firing Rates for Drifting Gratings')
plt.xlabel('Grating orientation (deg)')
degree_labels = np.arange(0, 360, 45)
plt.xticks(
    ticks=degree_labels, # 0, 45, 90, ..., 315
    labels=[f'{d}°' for d in degree_labels]
)
plt.ylabel('Average firing rate (Hz)')
plt.show()

In [ ]:
def visualize_grating_spike_rates(session, region="VISp", window_size=0.2):
    stim = "drifting_gratings" # Only get drifting_gratings data
    grating_presentations, units = get_units(session, stim=stim, region=region)
    grating_spikes = get_spikes(session, grating_presentations, units)

    grating_angles = get_grating_angles(grating_presentations)
    fast_spikes_clean = clean_grating_spike_data(grating_spikes, grating_angles, window_size=0.2)
    
    # Step 1: compute rate per (trial, unit)
    firing_rates = (
        fast_spikes_clean
        .groupby(['stimulus_presentation_id', 'unit_id'])
        .size()
        .div(window_size)
        .rename('rate_hz')
        .reset_index()
    )
    firing_rates = firing_rates.merge(grating_angles, on="stimulus_presentation_id")
    
    # Step 2: average across trials for each orientation
    orientation_tuning = (
        firing_rates.groupby('orientation')['rate_hz']
        .mean()
        .reset_index()
    )
    
    # Step 3: calculate 95% CIs
    summary = (
        firing_rates
        .groupby('orientation')['rate_hz']
        .agg(['mean', 'std', 'count'])
        .sort_values(by='orientation')
        .reset_index()
    )
    summary['ci95'] = 1.96 * summary['std'] / np.sqrt(summary['count'])
    
    # Step 4: plot avg firing rates against grating angle
    plt.plot(orientation_tuning['orientation'], orientation_tuning['rate_hz'], 'o-')
    plt.errorbar(
        summary['orientation'],
        summary['mean'],
        yerr=summary['ci95'],
        fmt='o-',
        color='navy',
        ecolor='crimson',
        capsize=4,
        label='mean ± 95% CI'
    )
    
    plt.title(f'{region} Firing Rates for Drifting Gratings (Specimen {session.specimen_name})')
    plt.xlabel('Grating orientation (deg)')
    degree_labels = np.arange(0, 360, 45)
    plt.xticks(
        ticks=degree_labels, # 0, 45, 90, ..., 315
        labels=[f'{d}°' for d in degree_labels]
    )
    plt.ylabel('Average firing rate (Hz)')
    plt.show()

In [ ]:
visualize_grating_spike_rates(session1, region="VISp")

In [ ]:
visualize_grating_spike_rates(session1, region="VISl")

In [ ]:
visualize_grating_spike_rates(session2, region="VISp")

In [ ]:
visualize_grating_spike_rates(session2, region="VISl")